# Compare raw and processed measurements

Explore the XJTU-SY raw and mean-removed example. Set the path to the exported dataset below. This notebook explicitly selects each file type; it does not apply any new processing.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from isa_phm import ISAWrapper

isa_json = Path('path/to/your-dataset.isa-phm.json').expanduser()
if not isa_json.is_file():
    raise FileNotFoundError(f'Set isa_json to your exported ISA-PHM JSON: {isa_json}')
wrapper = ISAWrapper(isa_json, data_root=isa_json.parent, strict_validation=False)
output_notebook()


## Select a study, sensor and measurement

This example selects the first available study, sensor and measurement. Change the indices to explore another one.


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
runs = assay.list_runs()
run = assay.run(runs[0].run_id)
print('Study:', study.title, '| Sensor:', assay.overview().sensor_alias, '| Run:', run.run_id)
display(pd.DataFrame([r.model_dump() for r in runs[:5]]))


## Inspect the recorded processing

The protocol and its parameter values describe the dataset's recorded processing. They are metadata; the Wrapper does not independently prove that the method was applied correctly.


In [ ]:
display([p.model_dump() for p in assay.list_protocols() if p.protocol_type == 'Processing Protocol'])
display(assay.list_processing_params(run_id=run.run_id))


## Load both files and compare provenance

Selecting `raw` and `processed` explicitly avoids the `auto` preference for processed data. Matching timestamps alone does not establish synchronization with other sensors.


In [ ]:
raw, raw_info = run.load_dataframe_with_meta(file_type='raw')
processed, processed_info = run.load_dataframe_with_meta(file_type='processed')
display(pd.DataFrame([raw_info.model_dump(), processed_info.model_dump()]))
print('Rows:', len(raw), len(processed))
print('Timestamps match:', raw['time'].equals(processed['time']))
display(pd.DataFrame({'time': raw['time'].head(), 'raw': raw['value'].head(), 'processed': processed['value'].head()}))


## Plot the same samples

The graph shows the first 1,000 samples for readability. The example's per-file mean removal shifts the signal vertically without changing its timestamps.


In [ ]:
unit = run.analysis_capabilities().measured_unit.value or 'unit unavailable'
plot = figure(title='Raw and processed signal — first 1,000 samples', x_axis_label='Recorded time', y_axis_label=f'Value ({unit})', width=900, height=350)
plot.line(raw['time'].iloc[:1000].to_numpy(), raw['value'].iloc[:1000].to_numpy(), legend_label='Raw', line_width=2)
plot.line(processed['time'].iloc[:1000].to_numpy(), processed['value'].iloc[:1000].to_numpy(), legend_label='Processed', line_width=2, line_color='orange')
show(plot)


## Compare built-in features

These features are calculated from each selected CSV, not read from the ISA-JSON. Mean removal forces the processed mean close to zero; RMS can change because it includes the signal's mean. No diagnosis or model is implied.


In [ ]:
raw_features = run.features(file_type='raw')
processed_features = run.features(file_type='processed')
display(pd.DataFrame([{'file_type': f['source']['resolved_file_type'], 'path': f['source']['file_path'], 'mean': f['mean'], 'rms': f['rms'], 'peak2peak': f['peak2peak'], 'excluded_samples': f['excluded_samples']} for f in (raw_features, processed_features)]))
print('Measured unit:', unit)
